# 02 — SMOTE / Resampling untuk Class Imbalance

`target_base` punya 35 kelas dengan distribusi sangat timpang — kelas
paling langka di train hanya 6 sampel (slam kecil/besar), sementara PASS
dan partscore umum punya ratusan. Notebook ini membandingkan 4 strategi
per model:

1. **none** — tidak ada penanganan imbalance (baseline polos)
2. **class_weight** — pembobotan kelas native (`class_weight="balanced"`
   untuk RandomForest & LightGBM; `sample_weight` dari
   `compute_sample_weight("balanced", ...)` untuk XGBoost karena
   `XGBClassifier` tidak punya `class_weight` multiclass native)
3. **smote** — oversampling sintetis (SMOTE) hanya pada train fold
4. **random_oversample** — duplikasi acak kelas minoritas (pembanding lebih
   sederhana dari SMOTE)

**Val/test set tidak pernah diresample** — resampling hanya mengubah
distribusi data training, evaluasi tetap di distribusi asli.


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-09" / "outputs" / "smote_resampling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_val, y_val = df_val[feature_cols].values, df_val["label"].values

class_counts = pd.Series(y_train).value_counts()
min_class_count = int(class_counts.min())
print(f"Train: {X_train.shape}, smallest class has {min_class_count} samples")
class_counts.sort_values().head(8)


Train: (7157, 164), smallest class has 5 samples


29     5
30     5
32     7
0      7
31    13
1     14
34    14
33    16
Name: count, dtype: int64

## 1. Base hyperparameters

Menggunakan default `configs/config.yaml` (n_estimators=300 di ketiga model) — bukan hasil tuning notebook 01, supaya efek resampling terisolasi dari efek tuning.

In [2]:
BASE_PARAMS = {
    "RandomForest": dict(
        n_estimators=300, max_depth=None, min_samples_split=5,
        random_state=42, n_jobs=-1,
    ),
    "XGBoost": dict(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, eval_metric="mlogloss", verbosity=0,
    ),
    "LightGBM": dict(
        n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1,
    ),
}

CLASSIFIERS = {
    "RandomForest": RandomForestClassifier,
    "XGBoost": XGBClassifier,
    "LightGBM": LGBMClassifier,
}

# Which models support class_weight natively for multiclass
NATIVE_CLASS_WEIGHT = {"RandomForest", "LightGBM"}


## 2. Resamplers

`k_neighbors` untuk SMOTE diadaptasi ke kelas terkecil supaya tidak error saat kelas minoritas < 6 sampel.

In [3]:
def make_smote(y):
    k = min(5, min_class_count - 1)
    k = max(k, 1)
    return SMOTE(k_neighbors=k, random_state=42)


def resample(strategy, X, y):
    if strategy == "smote":
        sampler = make_smote(y)
        return sampler.fit_resample(X, y)
    if strategy == "random_oversample":
        sampler = RandomOverSampler(random_state=42)
        return sampler.fit_resample(X, y)
    return X, y


## 3. Jalankan 4 varian x 3 model

In [4]:
VARIANTS = ["none", "class_weight", "smote", "random_oversample"]

all_results = []
timings = []
for model_name, params in BASE_PARAMS.items():
    for variant in VARIANTS:
        t0 = time.time()
        fit_kwargs = {}
        run_params = dict(params)

        if variant in ("smote", "random_oversample"):
            X_fit, y_fit = resample(variant, X_train, y_train)
        else:
            X_fit, y_fit = X_train, y_train

        if variant == "class_weight":
            if model_name in NATIVE_CLASS_WEIGHT:
                run_params["class_weight"] = "balanced"
            else:  # XGBoost: no native multiclass class_weight -> sample_weight
                fit_kwargs["sample_weight"] = compute_sample_weight("balanced", y_fit)

        clf = CLASSIFIERS[model_name](**run_params)
        clf.fit(X_fit, y_fit, **fit_kwargs)

        y_pred = clf.predict(X_val)
        y_proba = clf.predict_proba(X_val)
        res = evaluate(y_val, y_pred, y_proba, le, model_name=f"{model_name} [{variant}]")
        res["n_train_rows"] = int(len(y_fit))
        all_results.append(res)
        elapsed = time.time() - t0
        timings.append({"model": model_name, "variant": variant,
                         "seconds": round(elapsed, 1), "n_train_rows": len(y_fit)})
        print(f"{model_name:14s} [{variant:18s}] rows={len(y_fit):5d} "
              f"acc={res['accuracy']:.4f} f1_macro={res['f1_macro']:.4f} "
              f"f1_weighted={res['f1_weighted']:.4f} ({elapsed:.1f}s)")


RandomForest   [none              ] rows= 7157 acc=0.4801 f1_macro=0.1524 f1_weighted=0.4097 (4.8s)


RandomForest   [class_weight      ] rows= 7157 acc=0.4514 f1_macro=0.2693 f1_weighted=0.4673 (2.4s)


RandomForest   [smote             ] rows=50995 acc=0.4755 f1_macro=0.2539 f1_weighted=0.4608 (22.2s)


RandomForest   [random_oversample ] rows=50995 acc=0.4808 f1_macro=0.2170 f1_weighted=0.4394 (18.2s)


XGBoost        [none              ] rows= 7157 acc=0.5127 f1_macro=0.2685 f1_weighted=0.4685 (36.8s)


...

D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM       [smote             ] rows=50995 acc=0.5016 f1_macro=0.2288 f1_weighted=0.4615 (102.1s)


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM       [random_oversample ] rows=50995 acc=0.5127 f1_macro=0.2938 f1_weighted=0.4696 (136.4s)


## 4. Bandingkan hasil

In [5]:
comparison = compare_models(all_results)
comparison["n_train_rows"] = [r["n_train_rows"] for r in all_results]
comparison


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy,n_train_rows
model,,,,,,,,
RandomForest [none],0.480104,0.215072,0.146269,0.152354,0.409683,0.735160,0.824527,7157
RandomForest [class_weight],0.451402,0.260240,0.297077,0.269270,0.467313,0.712329,0.817352,7157
RandomForest [smote],0.475538,0.260411,0.272872,0.253867,0.460755,0.733203,0.829093,50995
RandomForest [random_oversample],0.480757,0.251265,0.208797,0.216961,0.439445,0.737117,0.842140,50995
XGBoost [none],0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010,7157
XGBoost [class_weight],0.490541,0.308807,0.285484,0.288748,0.484754,0.759948,0.856491,7157
XGBoost [smote],0.491194,0.292903,0.264622,0.265844,0.462687,0.752772,0.840183,50995
XGBoost [random_oversample],0.503588,0.324520,0.273588,0.288608,0.486359,0.752772,0.849967,50995
LightGBM [none],0.509459,0.357381,0.241093,0.261329,0.453988,0.746902,0.832355,7157


In [6]:
comparison.to_csv(OUT_DIR / "val_comparison.csv")

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "min_class_count_train": min_class_count,
    "variants": VARIANTS,
    "results": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy", "n_train_rows"]}
        for r in all_results},
    "timings": timings,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
print(f"Saved: {OUT_DIR / 'summary.json'}")


Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\smote_resampling\val_comparison.csv
Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\smote_resampling\summary.json


## 5. Kesimpulan

**Diperbarui 2026-07-10 setelah perbaikan split group-aware** — angka di
bawah ini MENGGANTIKAN kesimpulan sebelumnya.

Hasil val set (`experiments/2026-07-09/outputs/smote_resampling/val_comparison.csv`):

| Model | Variant | Accuracy | F1 macro | F1 weighted |
|---|---|---|---|---|
| LightGBM | class_weight | 52.1% | **0.309** | 0.480 |
| XGBoost | random_oversample | 50.4% | 0.289 | **0.486** |
| XGBoost | class_weight | 49.1% | 0.289 | 0.485 |
| XGBoost | none (baseline) | **51.3%** | 0.269 | 0.469 |
| LightGBM | none (baseline) | 50.9% | 0.261 | 0.454 |
| RandomForest | class_weight | 45.1% | 0.269 | 0.467 |
| RandomForest | none | 48.0% | 0.152 | 0.410 |
| RandomForest | smote | 47.6% | 0.254 | 0.461 |
| RandomForest | random_oversample | 48.1% | 0.217 | 0.439 |
| LightGBM | smote | 50.2% | 0.229 | 0.462 |
| XGBoost | smote | 49.1% | 0.266 | 0.463 |

- **SMOTE tetap konsisten paling buruk** di ketiga model — temuan ini
  BERTAHAN setelah split diperbaiki (bukan artefak kebocoran). Penjelasan
  yang sama masih berlaku: k-NN SMOTE tidak punya cukup tetangga asli
  untuk kelas dengan 5-16 sampel di train.
- **class_weight kini pemenang, bukan RandomOverSampler** — dengan split
  lama, XGBoost+RandomOverSampler unggul F1 macro (0.335). Dengan split
  yang diperbaiki, **LightGBM+class_weight** yang tertinggi (F1 macro
  0.309, accuracy 52.1% — malah lebih tinggi dari baseline manapun).
  RandomOverSampler masih solid (F1 weighted 0.486, tertinggi di
  notebook ini) tapi tidak lagi jelas-jelas terbaik di F1 macro.
- **RandomForest tanpa pembobotan apa pun (`none`) sangat buruk** di F1
  macro (0.152) — mengonfirmasi `class_weight="balanced"` (default di
  `src/models/random_forest.py`) penting untuk model ini, terlepas dari
  isu split.
- **Rekomendasi diperbarui**: LightGBM+class_weight dan
  XGBoost+RandomOverSampler sama-sama kandidat kuat tergantung prioritas
  (F1 macro vs F1 weighted/top-k) — validasi keduanya di test set.
